# Random-forest experiments

[English](04_random_forest.ipynb) · [简体中文](notebooks/zh-CN/04_random_forest.ipynb)

E1 predicts log2 half-life, while E2 predicts recall directly.


In [1]:
# Locate shared data and modules when running from a subdirectory.
from pathlib import Path
import os
import sys

_notebook_start = Path.cwd().resolve()
_notebook_parents = (_notebook_start, *_notebook_start.parents)
_project_root = next(
    (p for p in _notebook_parents
     if (p / "data/raw/settles.acl16.learning_traces.13m.csv.gz").is_file()),
    None,
)
if _project_root is None:
    _project_root = next(
        (p for p in _notebook_parents if (p / "experiment_pipeline.py").is_file()),
        _notebook_start,
    )
os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from experiment_pipeline import Pipeline
run = Pipeline()
globals().update(run.random_forest())


Raw file: D:\fujing123\Projects\Data-Analytic\HLR\data\raw\settles.acl16.learning_traces.13m.csv.gz
First 5 rows:


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,1.0,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,2,2
1,0.5,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,2,1
2,1.0,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,1,1
3,0.5,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,2,1
4,1.0,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,1,1


Reused saved duplicate check and cleaned dataset.


,raw_rows,exact_duplicates_removed,clean_rows
0,12854226,81,12854145


Model and EDA input: D:\fujing123\Projects\Data-Analytic\HLR\data\processed\settles.acl16.learning_traces.13m.dedup_v1.csv.gz


,split,start,stop_exclusive,rows
0,Train,0,9254984,9254984
1,Validation,9254984,11568730,2313746
2,Test,11568730,12854145,1285415


Split follows the original CSV row order. Shuffling is confined to Train during fitting.
Reused the feature cache matching the current data and split.
Cache:D:\fujing123\Projects\Data-Analytic\HLR\data\interim\takeaway_hlr_features_v1.dat；539.4 MiB
Base features: ['sqrt(1+history_correct)', 'sqrt(1+history_seen-history_correct)', 'bias']
Train lexemes:18,353; Validation rows with unseen lexemes:1,150; Test rows with unseen lexemes:560
Reused Train-only categorical index cache.
Train categories: 92,392 users, 6 learning languages, 4 UI languages.
Validation unseen users: 590,955
Loaded rf results: D:\fujing123\Projects\Data-Analytic\HLR\data\artifacts\rf-63a752907e48a563.joblib


# 5. Random-forest experiments

## 5.1 E1: RF-logh selection and full-data training

Use the three HLR -lex `x` features to predict `log2(h_observed)`. Fit six candidates on the same fixed 200,000 Train records, select by `MAE(p)` on the complete Validation split, and refit the selected configuration on all Train records. Evaluate Test in section 8.


In [2]:
from IPython.display import display
print("rf_candidate_results")
display(rf_candidate_results)
print("rf_e1_validation")
display(rf_e1_validation)
print("rf_e1_resource")
display(rf_e1_resource)


rf_candidate_results


,max_depth,min_samples_leaf,n_train,n_validation,fit_seconds,predict_seconds,MAE(p)
0,8,50,200000,2313746,0.399886,0.966054,0.160599
1,8,100,200000,2313746,0.364246,0.940063,0.160651
2,12,50,200000,2313746,0.410473,0.972090,0.160684
3,12,100,200000,2313746,0.379341,1.042311,0.160715
4,16,50,200000,2313746,0.400069,1.219036,0.160720
5,16,100,200000,2313746,0.486677,1.622735,0.160725


rf_e1_validation


,model,n_train,n_validation,MAE(p),RMSE(p),AUC,Spearman(h),n_spearman_h,n_h_interior,h_interior_share,Spearman(h)_interior,p_hat_at_bounds,h_hat_at_bounds
0,RF-logh (E1),9254984,2313746,0.160425,0.314653,0.548231,0.16364,2313746,1025825,0.443361,0.043372,0.240248,0.0


rf_e1_resource


,n_train,max_depth,min_samples_leaf,fit_seconds,predict_seconds,rss_before_GiB,peak_rss_GiB
0,9254984,8,50,64.693576,1.354101,0.682228,8.516899


## 5.2 E2: RF-p predicts recall directly

E2 uses the same 200,000 candidate-training rows as E1 and all Train rows for the final fit. Inputs are `[x_correct, x_wrong, bias, delta_days]` and the target is `p_target`. The model predicts `p_hat`; `h_hat` is inferred from `p_hat` and `delta_days` for diagnostics.


In [3]:
from IPython.display import display
print("rf_p_candidate_results")
display(rf_p_candidate_results)
print("rf_e2_validation")
display(rf_e2_validation)
print("rf_e2_resource")
display(rf_e2_resource)


rf_p_candidate_results


,max_depth,min_samples_leaf,n_train,n_validation,fit_seconds,predict_seconds,MAE(p)
0,12,50,200000,2313746,1.214223,1.882419,0.170564
1,16,50,200000,2313746,1.399276,1.743256,0.170566
2,12,100,200000,2313746,1.131645,1.811084,0.170703
3,16,100,200000,2313746,1.329878,1.694077,0.170711
4,8,50,200000,2313746,0.884785,1.353311,0.170763
5,8,100,200000,2313746,0.976290,2.052552,0.170875


rf_e2_validation


,model,n_train,n_validation,MAE(p),RMSE(p),AUC,Spearman(h),n_spearman_h,n_h_interior,h_interior_share,Spearman(h)_interior,p_hat_at_bounds,h_hat_at_bounds
0,RF-p (E2),9254984,2313746,0.169912,0.267622,0.608063,0.624224,2313746,1025825,0.443361,0.022707,0.000013,0.06882


rf_e2_resource


,n_train,max_depth,min_samples_leaf,fit_seconds,predict_seconds,rss_before_GiB,peak_rss_GiB
0,9254984,12,50,146.089527,1.583159,0.644188,8.490627
